In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("workspace.`ds-datasets`.billings") \
    .withColumn("index", F.monotonically_increasing_id())


df_calls = spark.table("workspace.`ds-datasets`.renewal_calls")


df_calls = df_calls \
    .filter(F.col("Analysed_Call") == "1")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date")) &
    (F.col("Call_Date") <= F.col("closed_date"))
).drop(df_calls["Co_Ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").saveAsTable("workspace.`ds-datasets`.billings_renewals")

In [0]:
df_final.groupBy("prospect_outcome").count().display()

In [0]:
df_final.select("co_ref").count()